## Import necessary python modules 

In [1]:
import json
import os
import pickle
import cv2
import numpy as np
import math
import time
import matplotlib.pyplot as plt
%matplotlib inline

%load_ext autoreload
%autoreload 2

#from tpmodules.spawn import spawnObject
from tpmodules.input_reader import VideoReader, ImageReader
from tpmodules.draw import Plotter3d, draw_poses
from tpmodules.parse_poses import parse_poses
from argparse import ArgumentParser

#### Cannot load fast pose extraction, switched to legacy slow implementation. ####


## Cube definition

In [2]:
# Cube with roof (house)
cube_vertices = np.float32([[0, 0, 0], [0, 1, 0], [1, 1, 0], [1, 0, 0],
                       [0, 0, 1], [0, 1, 1], [1, 1, 1], [1, 0, 1],
                       [0, 0.5, 1.4], [1, 0.5, 1.4]])
cube_vertices -= 0.5 # Translate house to the center of the world
cube_side_length = 140
cube_vertices *= cube_side_length # Upscale house 
cube_edges = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7), (4, 8), (5, 8), (6, 9), (7, 9), (8, 9)]

## Parse input arguments for skeleton extraction

In [3]:
if __name__ == '__main__':
    parser = ArgumentParser(description='Mini project parser')
    parser.add_argument('--video', help='Path to video file.', type=str, default='data/video-with-human.mp4')
    parser.add_argument('--bg_image', help='Path to the background image.', type=str, default='data/office-background.png')
    args, unknown = parser.parse_known_args()

## Determining Input source 

In [4]:
frame_provider = VideoReader(args.video)
is_video = True
bg_image = cv2.imread(args.bg_image)
bg_resized = None

## Declaration of constants and keyboard mappings

In [5]:
constant_delay = 50 # In ms. Should be set empirically depending on computer frame rate
delay = constant_delay

# Letter-keyboard associations
esc_code = 27 # 'Esc' button
p_code = 112 # Letter 'p' button
face_blur_code = 102 # Letter 'f' button
back_blur_code = 98 # Letter 'b' button
back_change_code = 115 # Letter 's' button
obj_hand_code = 111 # Letter 'o' button
space_code = 32

video_2d_window_name = 'Mini project'
fps = 1.0/(constant_delay*0.001)

# Face masking
face_blurring = 0
mask_size = 100

# Flags initialization
face_blur_flag = False  # Face blurring
back_blur_flag = False # Background blurring
back_change_flag = False # Background image change
obj_hand_flag = False # 3D object augmentation

skeleton_2d_poses_filename = 'data/human-skeleton-poses.bin' # Filename containing 2D skeleton poses

## Exercise area : Face blurring, background blurring, backround image change and 3D object addition

In [ ]:
constant_delay = 100 # 
delay = constant_delay
file_handler_2d = open(skeleton_2d_poses_filename, "rb" )

# For reference, skeleton joints are in the following order:
# ['centroid', neck', 'nose',  'l_sho', 'l_elb', 'l_wri', 'l_hip', 'l_knee', 'l_ank', 'r_sho', 'r_elb', 'r_wri', 'r_hip', 'r_knee', 'r_ank', 'r_eye', 'l_eye', 'r_ear', 'l_ear']

frame_count = 0

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('data/your-mini-project-output.mp4', fourcc, fps, (1280, 960))


for frame in frame_provider:
    if bg_resized is None:
        bg_resized = cv2.resize(bg_image, (frame.shape[1], frame.shape[0]))
    if frame is None: # If camera (or video) stop, exit the loop
        print('Frame is None because video/camera feed is empty or terminated')
        break
        
    frame_count += 1
    print(frame_count)
    poses_2d = pickle.load(file_handler_2d)
    
    pose = np.array(poses_2d[0][0:-1]).reshape((-1, 3)).transpose()
    
    final_frame = frame.copy()
    if poses_2d.size != 0 : # If skeleton/human exists in the current frame
        mask =None
        #################
        # Face blurring # 
        #################
        if face_blur_flag == True:
            print('Face blurring')
            h, w = 100, 100
            x = int(pose[0,1])  
            y = int(pose[1,1])  
            x1 = max(0, x - w)
            x2 = min(frame.shape[1], x + w)
            y1 = max(0, y - h)
            y2 = min(frame.shape[0], y + h)
            
            mask = frame[y1:y2, x1:x2]

            if mask.size != 0:
                blurred_img = cv2.GaussianBlur(mask, (25,25), 10)
                final_frame[y1:y2,x1:x2 ] = blurred_img
            
                        

        #######################
        # Background blurring #
        #######################
        if back_blur_flag == True:
            print('Background blurring')
            ### ADD YOUR CODE ###

            mask=np.zeros(frame.shape[:2], dtype=np.uint8)
            draw_poses(mask,poses_2d)
            kernel=np.ones((100,100),np.uint8)
            mask=cv2.dilate(mask,kernel,iterations=1)
            blurred_img = cv2.GaussianBlur(frame, (25, 25), 10)
            final_frame = np.where(mask[:, :, np.newaxis] == 255, final_frame,blurred_img)
            
                
        ##########################
        # Background picture set #
        ##########################
        if back_change_flag == True:
            print('Background image change')
            ### ADD YOUR CODE ###
            mask=np.zeros(frame.shape[:2], dtype=np.uint8)
            draw_poses(mask,poses_2d)
            kernel=np.ones((125,125),np.uint8)
            mask=cv2.dilate(mask,kernel,iterations=1)
            bg_image=cv2.imread("data/office-background.png")
            #print(frame.shape)
            bg_resized = cv2.resize(bg_image, (frame.shape[1], frame.shape[0]))
            final_frame = np.where(mask[:, :, np.newaxis] == (255,255,255), final_frame,bg_resized)
        ######################################    
        # Left-hand augmentation with object #
        ######################################
        if obj_hand_flag == True and pose[0, 5] != -1.0:
            print('3D object augmentation')

            x = int(pose[0, 5])
            y = int(pose[1, 5])
            fx = 593.47881447460395

            projected = []
            # angle = np.radians(frame_count * 10)
            # rvec = np.array([0, angle, 0], dtype=np.float32)
            # R, _ = cv2.Rodrigues(rvec)

            for v in cube_vertices:
                vx = v[2]
                vy = v[1]
                vz = -v[0]
                vx2 = -vy
                vy2 = vx
                vz2 = vz
                v_array = np.array([vx2, vy2, vz2])
                scale = fx / (fx + v_array[2])
                px = int(v_array[0] * scale + x)
                py = int(-v_array[1] * scale + y)
                projected.append((px, py))
            for (i, j) in cube_edges:
                cv2.line(final_frame, projected[i], projected[j], (255, 0, 0), 10)
    cv2.imshow(video_2d_window_name, final_frame) 
    out.write(final_frame)
    
    key = cv2.waitKey(delay) # Wait delay ms
    
    if key == esc_code: # Exit loop
        cv2.destroyAllWindows()
        break
        
    if key == face_blur_code: # face blurring
        face_blur_flag = not face_blur_flag
    if key == back_blur_code : # background blurring
        back_blur_flag = not back_blur_flag
    if key == back_change_code : # background photo change
        back_change_flag = not back_change_flag
    if key == obj_hand_code : # hand augmentation
        obj_hand_flag = not obj_hand_flag
        
    if key == p_code: # Pause or unpause
        if delay == constant_delay:
            delay = 0
        else:
            delay = constant_delay
    if delay == 0 or not is_video:  # allow to rotate 3D canvas while on pause
        key = 0
        while (key != p_code and key != esc_code and key != space_code):
            key = cv2.waitKey(10)
        if key == esc_code:
            cv2.destroyAllWindows()
            break
        else:
            delay = constant_delay
frame_provider.cap.release()
out.release()
print("Exiting")

1
3D object augmentation
2
3D object augmentation
3
3D object augmentation
4
3D object augmentation
5
3D object augmentation
6
3D object augmentation
7
3D object augmentation
8
3D object augmentation
9
3D object augmentation
10
3D object augmentation
11
3D object augmentation
12
3D object augmentation
13
3D object augmentation
14
3D object augmentation
15
3D object augmentation
16
3D object augmentation
17
3D object augmentation
18
3D object augmentation
19
3D object augmentation
20
3D object augmentation
21
3D object augmentation
22
3D object augmentation
23
3D object augmentation
24
3D object augmentation
25
3D object augmentation
26
3D object augmentation
27
3D object augmentation
28
3D object augmentation
29
3D object augmentation
30
3D object augmentation
31
3D object augmentation
32
3D object augmentation
33
3D object augmentation
34
3D object augmentation
35
3D object augmentation
36
3D object augmentation
37
3D object augmentation
38
3D object augmentation
39
3D object augmenta